In [1]:
import pandas as pd 
# Create the dataframe with your telecom data 
data = { 
 "customer_id": [1001, 1002, 1003, 1004, 1005], 
 "data_used_gb": [5.2, None, 7.8, 15.6, 3.4], 
 "calls_made": [25, 40, 32, 55, 18], 
 "revenue_inr": [180, 280, 210, None, 120], 
 "region": ["delhi", "Mumbai", "chennai", "DELHI", "Kolkata"], 
 "date": ["2025/09/25", "2025-09-25", "25-09-2025", "2025-09-25", "2025-09-25"] } 


In [3]:
# Create a DataFrame 
telecom_df = pd.DataFrame(data)
# Save to CSV 
telecom_df.to_csv("telecom_raw.csv", index=False) 


In [4]:
print(" telecom_raw.csv saved successfully.")
# 2) Write the automated ETL script (auto_etl.py) 


 telecom_raw.csv saved successfully.


This process is called ETL: 
Extract -> Transform -> Load 


In [5]:
#Importing libraries 
import os 
import time 
import pandas as pd 
import schedule 
from datetime import datetime 
#Setting up file paths 
RAW_PATH = "telecom_raw.csv" 
OUT_DIR = "output" 
OUT_PATH = os.path.join(OUT_DIR, "telecom_cleaned.csv") 
TMP_PATH = os.path.join(OUT_DIR, "telecom_cleaned.tmp.csv") 
LOG_PATH = os.path.join(OUT_DIR, "etl_run.log") 


In [6]:
#creates output folder 
os.makedirs(OUT_DIR, exist_ok=True) 


In [9]:
#Logs function 
def log(msg: str): 
 ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S") 
 with open(LOG_PATH, "a", encoding="utf-8") as f: f.write(f"[{ts}] {msg}\n") 
 print(f"[{ts}] {msg}") 


In [12]:
# Data cleaning function
def clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    # -------------------------------------------------------
    # 1) Standardize text columns – Fix text formatting
    # -------------------------------------------------------
    # If 'region' column exists:
    # - Convert values to string
    # - Remove extra spaces
    # - Convert to proper title case (north -> North)
    if "region" in df.columns:
        df["region"] = df["region"].astype(str).str.strip().str.title()

    # -------------------------------------------------------
    # 2) Fill missing numeric values with median
    # -------------------------------------------------------
    # Loop through important numeric columns
    # If column exists:
    # - Convert to numeric if needed
    # - Replace missing values with median
    for col in ["data_used_gb", "calls_made", "revenue_inr"]:
        if col in df.columns:

            # Convert to numeric if column type is not numeric
            if pd.api.types.is_numeric_dtype(df[col]) is False:
                df[col] = pd.to_numeric(df[col], errors="coerce")

            # Replace missing values with median
            df[col] = df[col].fillna(df[col].median())

    # -------------------------------------------------------
    # 3) Parse date column – Fix date problems
    # -------------------------------------------------------
    # Convert 'date' column to datetime format
    # If conversion fails, value becomes NaT
    # Replace missing dates with default date
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["date"] = df["date"].fillna(pd.Timestamp("2025-09-25"))

    # -------------------------------------------------------
    # 4) Remove duplicate rows
    # -------------------------------------------------------
    # If both customer_id and date columns exist:
    # - Remove duplicate records
    # - Keep the first occurrence
    # - Log how many rows were removed
    if {"customer_id", "date"}.issubset(df.columns):
        before = len(df)
        df = df.drop_duplicates(subset=["customer_id", "date"], keep="first")
        log(f"Deduplicated: removed {before - len(df)} duplicate row(s).")

    # -------------------------------------------------------
    # 5) Clip invalid values – Keep data realistic
    # -------------------------------------------------------
    # Ensure data usage is between 0 and 100 GB
    if "data_used_gb" in df.columns:
        df["data_used_gb"] = df["data_used_gb"].clip(lower=0, upper=100)

    # Ensure revenue is not negative
    if "revenue_inr" in df.columns:
        df["revenue_inr"] = df["revenue_inr"].clip(lower=0)

    # Return cleaned dataframe
    return df

In [13]:
# ETL JOB FUNCTION
def etl_job():
    try:
        # Log that the ETL process has started
        log("Starting ETL...")

        # Check if the raw CSV file exists
        # If the file is missing, log the error and stop the process
        if not os.path.exists(RAW_PATH):
            log(f"Raw file not found: {RAW_PATH}")
            return

        # -------------------------
        # EXTRACT PHASE
        # -------------------------
        # Read the raw telecom dataset from CSV
        df = pd.read_csv(RAW_PATH)

        # -------------------------
        # TRANSFORM PHASE
        # -------------------------
        # Apply data cleaning and transformation function
        # This function handles tasks like:
        # - Standardizing region names
        # - Filling missing values
        # - Fixing date formats
        # - Removing duplicates
        df = clean_frame(df)

        # -------------------------
        # LOAD PHASE
        # -------------------------
        # First write the cleaned data to a temporary file
        # This prevents half-written files if the program stops unexpectedly
        df.to_csv(TMP_PATH, index=False)

        # Atomically replace the old output file with the new cleaned file
        os.replace(TMP_PATH, OUT_PATH)

        # Log successful completion and number of rows written
        log(f"ETL completed successfully. Rows written: {len(df)}")

    except Exception as e:
        # Catch any error during ETL and log the failure message
        log(f"ETL failed: {e}")

In [14]:
# -----------------------------
# SCHEDULING CONFIGURATION
# -----------------------------
# This section controls when the ETL job runs automatically.

import time
import schedule

# Clear any previously scheduled jobs
schedule.clear()

# -----------------------------
# DEMO MODE
# -----------------------------
# Run ETL every 10 seconds so students can quickly see the automation working
schedule.every(10).seconds.do(etl_job)

# -----------------------------
# PRODUCTION MODE (Examples)
# -----------------------------
# Uncomment one of these for real deployment

# Run ETL every day at 09:00 AM
# schedule.every().day.at("09:00").do(etl_job)

# Run ETL at the start of every hour
# schedule.every().hour.at(":00").do(etl_job)

# -----------------------------
# CLASSROOM DEMO EXECUTION
# -----------------------------
# For demonstration we run the scheduler only 3 times
runs = 3

print("Scheduler started...")

for i in range(runs):
    # Execute pending scheduled jobs
    schedule.run_pending()

    # Wait 10 seconds before checking again
    time.sleep(10)

print("Done. Scheduler exited after", runs, "runs.")

Scheduler started...
[2026-03-12 10:28:06] Starting ETL...
[2026-03-12 10:28:06] Deduplicated: removed 0 duplicate row(s).
[2026-03-12 10:28:06] ETL completed successfully. Rows written: 5
[2026-03-12 10:28:16] Starting ETL...
[2026-03-12 10:28:16] Deduplicated: removed 0 duplicate row(s).
[2026-03-12 10:28:16] ETL completed successfully. Rows written: 5
Done. Scheduler exited after 3 runs.
